# Person B — EDA and HMM preparation

This notebook follows the Person B task list: dataset inventory, missing values and timestamp gaps, network size/connectivity, GPS plots, data-quality notes, and a Folium map with GPS points, road network and reference route. It then smoke-tests the Viterbi decoder on a toy graph.

Run from the repository root. Raw data is expected under `data/raw/train/<trip_id>/` and `data/raw/student_test/<trip_id>/`.

In [ ]:
from pathlib import Path
import sys, json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.sparse import csr_matrix
from scipy.sparse.csgraph import connected_components

ROOT = Path.cwd()
if not (ROOT / 'src').exists():
    raise RuntimeError('Open this notebook from the repository root (the folder containing src/).')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.data.loaders import list_trips, load_trip
from src.viz.folium_map import create_trip_map
from src.hmm.viterbi import demo_toy_graph
DATA_ROOT = ROOT / 'data' / 'raw'
print('Repository:', ROOT)
print('Raw data folder exists:', DATA_ROOT.exists())

## 1. Dataset inventory

In [ ]:
def inventory_split(split):
    folder = DATA_ROOT / split
    if not folder.exists():
        return []
    rows = []
    for trip_id in list_trips(folder):
        trip = load_trip(folder, trip_id)
        rows.append({
            'split': split, 'trip_id': trip_id,
            'gps_points': len(trip['track']),
            'nodes': len(trip['nodes']), 'arcs': len(trip['arcs']),
            'reference_route_arcs': None if trip['route'] is None else len(trip['route']),
            'has_route': trip['route'] is not None
        })
    return rows

inventory = pd.DataFrame(inventory_split('train') + inventory_split('student_test'))
display(inventory.head(10))
if not inventory.empty:
    display(inventory.groupby('split')[['gps_points','nodes','arcs','reference_route_arcs']].sum(numeric_only=True))
else:
    print('No trips found. Check data/raw/train and data/raw/student_test.')

## 2. Missing values and timestamp gaps

In [ ]:
def inspect_trip_quality(trip):
    track = np.asarray(trip['track'])
    times = track[:, 2] if track.shape[1] >= 3 else np.array([])
    dt = np.diff(times) if len(times) else np.array([])
    return {
        'trip_id': trip['trip_id'],
        'gps_rows': len(track),
        'missing_track_values': int(pd.isna(track).sum()),
        'non_increasing_time_gaps': int((dt <= 0).sum()) if len(dt) else 0,
        'gaps_over_60s': int((dt > 60).sum()) if len(dt) else 0,
        'max_gap_seconds': float(np.max(dt)) if len(dt) else np.nan,
        'duplicate_timestamps': int(pd.Series(times).duplicated().sum()) if len(times) else 0
    }

quality_rows = []
for split in ['train', 'student_test']:
    folder = DATA_ROOT / split
    if folder.exists():
        ids = list_trips(folder)
        for tid in ids[:10]:  # inspect up to 10 trips per split for a quick EDA pass
            quality_rows.append(inspect_trip_quality(load_trip(folder, tid)))
quality = pd.DataFrame(quality_rows)
display(quality)

## 3. Road-network size and connectivity

In [ ]:
def network_stats(trip):
    arcs = np.asarray(trip['arcs'], dtype=int)
    n = len(trip['nodes'])
    if len(arcs) == 0:
        return {'trip_id': trip['trip_id'], 'nodes': n, 'arcs': 0, 'components_weak': n}
    rows, cols = arcs[:, 0], arcs[:, 1]
    graph = csr_matrix((np.ones(len(arcs)), (rows, cols)), shape=(n, n))
    n_comp, labels = connected_components(graph, directed=True, connection='weak')
    return {'trip_id': trip['trip_id'], 'nodes': n, 'arcs': len(arcs),
            'components_weak': int(n_comp), 'largest_component_nodes': int(np.bincount(labels).max()),
            'mean_out_degree': float(np.bincount(rows, minlength=n).mean())}

if not inventory.empty:
    sample_split = 'train' if (DATA_ROOT/'train').exists() else 'student_test'
    sample_id = int(inventory[inventory.split == sample_split].iloc[0].trip_id)
    sample_trip = load_trip(DATA_ROOT/sample_split, sample_id)
    stats = network_stats(sample_trip)
    display(pd.DataFrame([stats]))
else:
    sample_trip = None
    print('Network analysis skipped: no trip data found.')

## 4. GPS track plot

In [ ]:
if sample_trip is not None:
    track = sample_trip['track']
    plt.figure(figsize=(8, 6))
    plt.plot(track[:, 0], track[:, 1], '-o', markersize=2, linewidth=1)
    plt.xlabel('Longitude'); plt.ylabel('Latitude')
    plt.title(f'GPS observations — trip {sample_trip["trip_id"]}')
    plt.grid(True); plt.axis('equal'); plt.show()

## 5. Folium map: GPS points, road network and reference route

In [ ]:
if sample_trip is not None:
    map_path = ROOT / 'maps' / 'person_b' / f'trip_{sample_trip["trip_id"]}_eda.html'
    trip_map = create_trip_map(sample_trip, output_path=map_path, max_road_edges=5000)
    print('Saved interactive map:', map_path)
    display(trip_map)
else:
    print('Map skipped: no trip data found.')

## 6. Initial data-quality notes

Review the tables above and record issues actually observed. Common checks include missing coordinates, duplicate or non-increasing timestamps, long timestamp gaps, GPS points far from the network, disconnected network components, and reference routes that do not cover the full GPS track. Do not report an issue unless the output confirms it.

In [ ]:
if not quality.empty:
    print('Trips inspected:', len(quality))
    print('Trips with missing track values:', int((quality.missing_track_values > 0).sum()))
    print('Trips with non-increasing timestamps:', int((quality.non_increasing_time_gaps > 0).sum()))
    print('Trips with gaps > 60 seconds:', int((quality.gaps_over_60s > 0).sum()))
if sample_trip is not None and sample_trip.get('route') is not None:
    route = sample_trip['route']
    arcs = sample_trip['arcs']
    valid = 0
    for a, b in zip(route[:-1], route[1:]):
        if 0 <= a < len(arcs) and 0 <= b < len(arcs) and arcs[a, 1] == arcs[b, 0]:
            valid += 1
    total = max(0, len(route)-1)
    print(f'Reference route continuity: {valid}/{total} = {100*valid/total if total else 100:.2f}%')

## 7. Viterbi decoder smoke test on a toy graph

In [ ]:
path, best_log_probability = demo_toy_graph()
state_names = ['A', 'B', 'C']
print('Decoded states:', [state_names[i] for i in path])
print('Best log probability:', best_log_probability)
assert len(path) == 3
print('Viterbi toy-graph test passed.')

## 8. HMM integration notes

The repository's candidate-feature interface is documented in `src/features/candidates.py`: it creates one row per GPS point and candidate arc, with `trip_id`, `point_idx`, `candidate_arc_id`, `dist_to_arc`, `bearing_diff`, `speed`, `arc_length`, `proj_pos`, and `dt_prev` among its columns. The Viterbi decoder in `src/hmm/viterbi.py` accepts log-emission scores shaped `(time_steps, states)` and a log-transition matrix shaped `(previous_states, next_states)`. A later integration step must group candidate rows by `point_idx`, construct emissions and legal transitions for each trip, and decode one state per observation. The Viterbi toy test does not yet constitute a complete road-matching HMM.